# Retained multi-seed ForTy architecture pilot

Select an available **T4 GPU** under **Runtime > Change runtime type** and run
cells in order. This notebook compares CNN + selective SSM against CNN-only
on real ForTy TFRecords using **three model seeds** and **one fixed shard cohort**.
It saves best/last checkpoints and evidence to **Google Drive**.

**Drive authorization:** allow notebook credential access and consent to every
requested permission (check all boxes). If Colab reports `credential propagation
was unsuccessful`, rerun the Drive cell and complete those permissions. Use the
same Google account for Colab and the intended Drive. See the
[maintainer guidance](https://github.com/googlecolab/colabtools/issues/6089).
Staging and training remain blocked until the persistent-study cell succeeds.

Defaults: seeds 42/43/44, selection seed 2026, 8 training / 4 validation /
8 test shards, batch 8 and 10 epochs per model. This is a reduced-data pilot,
not the full ForTy benchmark. CNN-only removes both SSM blocks and temporal
order information; parameter counts differ, so this is a component ablation.

All six training runs must validate before test evaluation starts. Model
selection uses validation macro F1. Completed stages are checked before reuse;
interrupted training is preserved, then restarted from epoch one. Checkpoints
contain weights, not optimizer state. Free GPU duration/availability varies.

The publisher's integer label mapping and Unknown protocol are still
[awaiting clarification](https://github.com/google-deepmind/forest_typology/issues/2).
The primary report uses numeric IDs 0–8 and explicitly declared metric
conventions. It does not publish semantic forest-class claims or promise a
positive architecture effect. The separate 4.80× result is synthetic batch-8
versus batch-1 inference throughput, excluding file IO.

In [ ]:
import datetime
import json
import os
from pathlib import Path
import re
import signal
import subprocess
import sys
import uuid

PERSISTENT_STUDY_READY = False
SOURCE_COMMIT = "7f55b2c30c830693873e1fe1628c8e52208ed57d"
REPOSITORY = "https://github.com/CCC917111/geospatial-mamba-deforestation.git"
STUDY_ID = "forty-architecture-pilot-v1"
SEEDS = [42, 43, 44]
SELECTION_SEED = 2026
TRAIN_SHARDS = 8
VAL_SHARDS = 4
TEST_SHARDS = 8
EPOCHS = 10
BATCH_SIZE = 8
PRECISION = "mixed_float16"
MAX_TRAIN_RUNS = None  # Optional per-session limit; None runs all remaining models.
STOP_AFTER_TRAINING = False
RECOVER_LOCK = False  # Enable only after confirming the previous runtime has stopped.
DOWNLOAD_BEST_CHECKPOINTS = False  # Weights remain in Drive even when False.
SOURCE_DATA = "gs://forest_typology/forty_v1/1.0.0"

if not re.fullmatch(r"[A-Za-z0-9_-]+", STUDY_ID):
    raise ValueError("STUDY_ID must contain only letters, digits, underscore or hyphen")
WORKSPACE = Path("/content/geospatial-research")
WORKSPACE.mkdir(parents=True, exist_ok=True)
SOURCE_ROOT = WORKSPACE / ("source-" + uuid.uuid4().hex[:8])
CACHE_ROOT = WORKSPACE / "cache" / STUDY_ID
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / "cache/matplotlib"))
os.environ.setdefault("KERAS_HOME", str(WORKSPACE / "cache/keras"))
os.environ.setdefault("XDG_CACHE_HOME", str(WORKSPACE / "cache"))
print({"study": STUDY_ID, "model_seeds": SEEDS, "selection_seed": SELECTION_SEED,
       "train_val_test_shards": [TRAIN_SHARDS, VAL_SHARDS, TEST_SHARDS],
       "epochs_per_run": EPOCHS, "batch_size": BATCH_SIZE})

In [ ]:
subprocess.run(["git", "clone", "--no-checkout", REPOSITORY, str(SOURCE_ROOT)], check=True)
subprocess.run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=SOURCE_ROOT, check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SOURCE_ROOT, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=SOURCE_ROOT, text=True)
if head != SOURCE_COMMIT or dirty:
    raise RuntimeError("Pinned source mismatch or modified checkout")
DRIVER_ENV = {**os.environ, "PYTHONPATH": str(SOURCE_ROOT / "src")}
# Probe before mounting Drive or downloading selected records.
probe = "from mamba_forest.research import RUNTIME_PROBE; exec(RUNTIME_PROBE)"
subprocess.run([sys.executable, "-c", probe, "gpu"], cwd=SOURCE_ROOT,
               env=DRIVER_ENV, check=True)

In [ ]:
PERSISTENT_STUDY_READY = False
from google.colab import drive

try:
    drive.mount("/content/drive")
except Exception as error:
    raise RuntimeError(
        "Google Drive authorization failed. Rerun this cell, allow notebook "
        "credential access, select the account for this Drive and consent to "
        "all requested permissions (check every permission box). If no prompt "
        "appears, refresh Colab using that same account and check browser "
        "pop-up/cookie settings. Complete this cell before staging or training. "
        "Upstream guidance: https://github.com/googlecolab/colabtools/issues/6089"
    ) from error
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not os.path.ismount("/content/drive") or not DRIVE_ROOT.is_dir():
    raise RuntimeError("Drive did not expose MyDrive. Complete Drive mounting before continuing.")
STUDY_ROOT = DRIVE_ROOT / "geospatial-mamba/research" / STUDY_ID
STUDY_ROOT.mkdir(parents=True, exist_ok=True)
DATA_MANIFEST = STUDY_ROOT / "selected_data.json"
BASE_COMMAND = [sys.executable, "-m", "mamba_forest.research",
                "--run-root", str(STUDY_ROOT), "--data-root", str(CACHE_ROOT),
                "--data-manifest", str(DATA_MANIFEST), "--data-kind", "real_forty",
                "--seeds", *map(str, SEEDS), "--selection-seed", str(SELECTION_SEED),
                "--train-shards", str(TRAIN_SHARDS), "--val-shards", str(VAL_SHARDS),
                "--test-shards", str(TEST_SHARDS), "--epochs", str(EPOCHS),
                "--batch-size", str(BATCH_SIZE), "--precision", PRECISION, "--resume"]
if RECOVER_LOCK:
    BASE_COMMAND.append("--recover-lock")
subprocess.run(BASE_COMMAND, cwd=SOURCE_ROOT, env=DRIVER_ENV, check=True)
print("Persistent study:", STUDY_ROOT)
print("Frozen protocol:", json.loads((STUDY_ROOT / "research_spec.json").read_text()))
PERSISTENT_STUDY_READY = True

Keep the same **STUDY_ID, source commit and scientific settings** to continue
on another runtime. The driver verifies source, software versions, data
identity and completed checkpoints. Changing the cohort, seeds, precision,
epoch budget or model settings requires a new study ID.

Staging downloads only the chosen 20 shards to local disk, with a 16 GiB byte
budget and a disk-space check. The same published filenames and 1,024-shard
namespaces are retained. New runtimes restage missing files and require the
original recorded hashes. Source records remain local; checkpoints and
metadata go to Drive. Stage failures preserve their logs and partial files.

Use only one active runtime per study. A normal interruption is cleaned up;
if a hard runtime loss leaves `.research.lock`, confirm the old runtime has
stopped before setting `RECOVER_LOCK=True`. If the tiny `.research.recovery`
acquisition marker remains, stop the old runtime and remove only that marker.
Do not delete checkpoints or identity receipts to bypass a failed validation.

In [ ]:
def execute_with_cleanup(argv, cwd, log, env=None):
    process = subprocess.Popen(argv, cwd=cwd, stdout=log,
                               stderr=subprocess.STDOUT, text=True,
                               start_new_session=True, env=env)
    try:
        return_code = process.wait()
    finally:
        # First allow the driver to record interruption and release its study lock.
        try:
            os.killpg(process.pid, signal.SIGINT)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            pass
        # The process group includes the driver and its current GPU worker.
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            pass
        finally:
            # Reap descendants even if the driver has already exited.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, argv)



def export_metadata():
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    destination = STUDY_ROOT.parent / (STUDY_ID + "-metadata-" + stamp + "-" + uuid.uuid4().hex[:6])
    subprocess.run([sys.executable, str(SOURCE_ROOT / "scripts/archive_profiles.py"),
                    "--run-root", str(STUDY_ROOT), "--output-dir", str(destination)],
                   cwd=SOURCE_ROOT, env=DRIVER_ENV, stdout=subprocess.DEVNULL, check=True)
    print("Metadata ZIP:", destination / "metadata.zip")
    print("Best/last checkpoints remain in Drive:", STUDY_ROOT)
    return destination / "metadata.zip"

In [ ]:
if (not globals().get("PERSISTENT_STUDY_READY", False)
        or not os.path.ismount("/content/drive") or not DRIVE_ROOT.is_dir()):
    raise RuntimeError("Complete the Drive authorization and persistent-study cell before staging or training.")

staging = [sys.executable, "-m", "mamba_forest.stage_data",
           "--source-root", SOURCE_DATA, "--cache-root", str(CACHE_ROOT),
           "--manifest", str(DATA_MANIFEST), "--selection-seed", str(SELECTION_SEED),
           "--train-shards", str(TRAIN_SHARDS), "--val-shards", str(VAL_SHARDS),
           "--test-shards", str(TEST_SHARDS), "--max-bytes", str(16 * 1024**3)]
staging_log = STUDY_ROOT / "logs" / ("staging-" + uuid.uuid4().hex[:8] + ".log")
staging_log.parent.mkdir(exist_ok=True)
print("Staging selected records; log:", staging_log)
try:
    with staging_log.open("x") as log:
        execute_with_cleanup(staging, cwd=SOURCE_ROOT, log=log, env=DRIVER_ENV)
except subprocess.CalledProcessError:
    print(staging_log.read_text()[-6000:])
    print("Run the download cell below to retain the staging failure log.")
    raise
manifest = json.loads(DATA_MANIFEST.read_text())
print("Verified staged shards:", len(manifest["files"]))
print("Selected source bytes:", sum(item["bytes"] for item in manifest["files"]))

In [ ]:
if (not globals().get("PERSISTENT_STUDY_READY", False)
        or not os.path.ismount("/content/drive") or not DRIVE_ROOT.is_dir()):
    raise RuntimeError("Complete the Drive authorization and persistent-study cell before staging or training.")

command = [*BASE_COMMAND, "--execute"]
if MAX_TRAIN_RUNS is not None:
    command += ["--max-train-runs", str(MAX_TRAIN_RUNS)]
if STOP_AFTER_TRAINING:
    command.append("--stop-after-training")
log_path = STUDY_ROOT / "logs" / ("driver-" + uuid.uuid4().hex[:8] + ".log")
log_path.parent.mkdir(exist_ok=True)
print("Running retained study; progress log:", log_path)
try:
    with log_path.open("x") as log:
        execute_with_cleanup(command, cwd=SOURCE_ROOT, log=log, env=DRIVER_ENV)
except subprocess.CalledProcessError:
    print(log_path.read_text()[-6000:])
    print("Run the download cell below to retain the available metadata.")
    raise
finally:
    METADATA_ZIP = export_metadata()

In [ ]:
state_path = STUDY_ROOT / "research_state.json"
if state_path.is_file():
    state = json.loads(state_path.read_text())
    print("Execution state:", state["state"])
    pending = [name for name, value in state["stages"].items()
               if name.startswith("train_") and value["state"] != "completed"]
    print("Remaining training stages:", pending)
inventory_path = STUDY_ROOT / "checkpoint_inventory.json"
if inventory_path.is_file():
    inventory = json.loads(inventory_path.read_text())
    print("Verified training runs:", inventory["completed_training_runs"],
          "/", inventory["required_training_runs"])
report_path = STUDY_ROOT / "research_report.json"
if report_path.is_file():
    report = json.loads(report_path.read_text())
    print("Report status:", report["status"])
    print(json.dumps(report["summary"], indent=2))
    print("Paired deltas (percentage points):",
          json.dumps(report["summary"]["paired_delta_percentage_points"], indent=2))
else:
    print("The pilot is not complete. Continue the same frozen study to finish remaining stages.")

A completed default study retains **six validation-selected best checkpoints**
and six last checkpoints, configs, full epoch CSVs, exact split manifests,
selection receipts, preflights, source/runtime/package identities, logs and
per-seed test confusion matrices. `checkpoint_manifest.json` records sizes and
SHA-256 hashes. The neutral report uses macro F1 over all nine IDs, mIoU over
classes with true/predicted support, pixel accuracy and paired seed deltas.
Three-seed means/sample standard deviations are descriptive, not proof of
statistical significance. Selected validation/test shards are subsets and do
not independently verify geographic separation.

The metadata ZIP excludes checkpoints, source arrays and TFRecords. They are
not discarded: weights remain in Drive. The optional toggle can download each
best checkpoint separately; large files are never combined into an unbounded
ZIP. Only successfully written Drive files can survive runtime reclamation.

For a complete best/last weight backup with matching model configurations, use
the separate [CPU checkpoint backup notebook](https://colab.research.google.com/github/CCC917111/geospatial-mamba-deforestation/blob/main/notebooks/colab_checkpoint_backup.ipynb).
It verifies actual source bytes and prepares bounded archives. Verify the
downloaded files on your computer to complete the local backup.

In [ ]:
from google.colab import files

if "METADATA_ZIP" not in globals():
    METADATA_ZIP = export_metadata()
files.download(str(METADATA_ZIP))
if DOWNLOAD_BEST_CHECKPOINTS:
    import hashlib
    import shutil

    inventory = json.loads((STUDY_ROOT / "checkpoint_inventory.json").read_text())
    destination = WORKSPACE / ("checkpoint-downloads-" + uuid.uuid4().hex[:8])
    destination.mkdir()
    for run in inventory["runs"]:
        checkpoint = (STUDY_ROOT / run["best"]["path"]).resolve()
        if not checkpoint.is_relative_to(STUDY_ROOT.resolve()) or not checkpoint.name.endswith(".weights.h5"):
            raise ValueError("Invalid checkpoint path in inventory")
        digest = hashlib.sha256()
        with checkpoint.open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        if digest.hexdigest() != run["best"]["sha256"]:
            raise ValueError("Checkpoint changed after verification")
        if checkpoint.stat().st_size >= 32 * 1024**2:
            print("Keep this checkpoint in Drive; it exceeds the archive size limit:", checkpoint)
        else:
            name = f"{run['variant']}-seed{run['seed']}-best.weights.h5"
            target = destination / name
            shutil.copyfile(checkpoint, target)
            files.download(str(target))
